In [5]:
import json
import glob
import re
import os.path as op
import requests
# base_url = "http://172.18.167.133:3889/api/v1/smear_analysis" # local
base_url = "http://192.168.31.188:3889/api/v1/smear_analysis" # formal

In [6]:
# 读取图片和坐标信息
root = r"./data20250901"
path = op.join(root, "Images")
images = glob.glob(op.join(path, "*.jpg"))
pattern = re.compile(r"Pos\[(\d+)\]\[(\d+)\]")
rows = [int(pattern.findall(x)[0][1]) for x in images]
cols = [int(pattern.findall(x)[0][0]) for x in images]
data = list(zip(rows, cols, images))
data = [
    (row, col, img)
    for row, col, img in data
]
data.sort(key=lambda x: (x[0], x[1]))
num_rows = max([x[0] for x in data]) + 1
num_cols = max([x[1] for x in data]) + 1

test_json = json.load(open(r'./test.json', encoding='utf-8'))
map_dict = {}
for one in test_json:
    map_dict[(one['row_index'], one['col_index'])] = (one['x_topleft'], one['y_topleft'])

imageinfo = op.join(root, "picInfo.json")
with open(imageinfo, 'r') as f:
    image_info = json.load(f)
    info = image_info.copy()
    info.pop('base_pixel', None)
    image_info = {
        (x['index_y'], x['index_x']): [
            int(x['left_x']),
            int(x['left_y']),
            int(x['top_x']),
            int(x['top_y']),
        ]
        for x in image_info['base_pixel']
    }

data = {
    (x[0], x[1]): x[2] for x in data
}

ValueError: max() arg is an empty sequence

In [ ]:
# 1.拨片任务创建
url = f"{base_url}/create_task"
d = {
    "num_rows": num_rows,
    "num_cols": num_cols,
    "tile_width": info['pixel_width'],
    "tile_height": info['pixel_height'],
    'smear_type': "BM",
    'dpi': 138430
}
response = requests.post(url, json=d)
if response.status_code != 200:
    raise Exception(f"Failed to create task: {response.text}")
json_data = response.json()
print(json_data)
task_id = json_data['task_id']

In [ ]:
# 2.拼图块上传
import requests
import concurrent.futures
from tqdm import tqdm
import os


# 上传单个文件的函数
def upload_file(item):
    row, col = item
    if row >= num_rows or col >= num_cols:
        return

    path = data[(row, col)]

    form_data = {
        "task_id": task_id,
        "row_index": row,
        "col_index": col
    }

    try:
        with open(path, 'rb') as f:
            imgdata = f.read()

        files = {
            "tile_image": (os.path.basename(path), imgdata)
        }

        response = requests.post(
            f"{base_url}/upload_tile",
            data=form_data,
            files=files,
            timeout=5
        )

        # 可以根据需要返回结果或日志
        return response.json()
        # return 200
    except Exception as e:
        return f"Error: {e}"


# 多线程执行上传
def run_uploads_parallel(items):
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:  # 控制并发数
        futures = [executor.submit(upload_file, item) for item in items]

        # 使用 tqdm 显示进度条
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Uploading"):
            # print(future.result())  # 打印每个上传的结果
            pass
        # 如果需要处理结果，可以使用 futures 或 future.result()


run_uploads_parallel(data)

In [ ]:
# 3.更新图块坐标信息
tiles_msg = []
for item in data:
    position_x, position_y = map_dict[item]
    tiles_msg.append({
        'row_index': item[0],
        'col_index': item[1],
        'position_x': position_x,
        'position_y': position_y
    })
update_coordinates = f'{base_url}/update_coordinates'
result = requests.post(update_coordinates, json={
    'task_id': task_id,
    'tiles_msg': tiles_msg
})
print(result.json())

In [ ]:
# 4.检查缺失拼图块
check_image_url = f"{base_url}/check_missing_tiles"
params = {
    "task_id": task_id,
    'row_id': num_rows,
    'max_ret_num': 100,
}
response = requests.post(check_image_url, json=params)
if response.status_code != 200:
    raise Exception(f"Failed to check image: {response.text}")
print(response.json())

In [ ]:
# 5.任务状态检查
get_result_url = f"{base_url}/check_task_status"
params = {
    "task_id": task_id,
}
response = requests.post(get_result_url, json=params)
print(response.json())

In [ ]:
# 6.获取拨片上面的细胞
get_result_url = f"{base_url}/get_task_result"
params = {
    "task_id": task_id,
    'roi_xmax': 1000,
    'roi_ymax': 1000,
    'index_offset': 1,
    'request_task_num': 10
}
response = requests.post(get_result_url, json=params)
if response.status_code != 200:
    raise Exception(f"Failed to get result: {response.text}")
print(response.json())
if response.json().get('match_result'):
    print(response.json())

In [ ]:
# 7. 百倍拍摄任务列表生成
get_task_x100_list = f"{base_url}/roi_selection"
params = {
    "task_id": task_id,
    "view_width": 2048,
    "view_height": 2448,
    "target_list": [
        {
            "type": "BM_WBC",
            "count": 200
        }
    ],
    "index_offset": 0,
    "request_task_num": 100
}
response = requests.post(get_task_x100_list, json=params)
print(len(response.json().get('task_list')))

In [ ]:
# 8.单张细胞图像识别（定位+分类）
analyze_cell_image_url = f"{base_url}/analyze_cell_image"
form_data = {
    "task_id": task_id,
    "position_xmin": 0,
    "position_ymin": 0,
    "position_xmax": 100,
    "position_ymax": 100,
    "dpi": 368116,
    "algorithm_type": "BM_WBC",
    "edge_cell_filter": False
}
one_img = './1.jpg'
with open(one_img, 'rb') as f:
    imgdata = f.read()
files = {
    "tile_image": (os.path.basename(path), imgdata)
}
response = requests.post(
    analyze_cell_image_url,
    data=form_data,
    files=files
)
print(response.json().get('task_list'))